In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from opendartreader import OpenDartReader

# API 및 환경 세팅
load_dotenv()
api_key = os.environ.get('DART_API_KEY')
dart = OpenDartReader(api_key)

# 매핑 테이블에서 삼성전자 고유번호(corp_code) 추출
current_path = os.getcwd()
root_path = os.path.dirname(current_path)
csv_path = os.path.join(root_path, 'data', 'kospi_top50_mapping.csv')
mapping_df = pd.read_csv(csv_path, dtype={'corp_code': str})

raw_code = mapping_df.loc[mapping_df['corp_name'] == '삼성전자', 'corp_code'].values[0]
samsung_corp_code = str(raw_code).zfill(8)
print(f"✅ 삼성전자 DART 고유번호: {samsung_corp_code}")

# DART 재무제표 API 호출 (finstate)
# bsns_year: 사업연도 (현재 시점 기준 가장 최신 결산연도인 2025년 적용)
# reprt_code: '11011' (사업보고서)
target_year = '2025'
print(f"🚀 {target_year}년도 삼성전자 재무제표 수집 중...")

samsung_finstate = dart.finstate(samsung_corp_code, target_year, reprt_code='11011')

# 수집된 결과 확인 (핵심 컬럼만 필터링)
# account_nm(계정명), fs_nm(재무제표명), thstrm_amount(당기금액), frmtrm_amount(전기금액)
if samsung_finstate is not None and not samsung_finstate.empty:
    display_cols = ['fs_nm', 'account_nm', 'thstrm_amount', 'frmtrm_amount']
    print("\n✅ 수집 완료! 주요 재무 계정 내역:")
    display(samsung_finstate[display_cols].head(10))
else:
    print("❌ 해당 연도의 재무 데이터가 아직 공시되지 않았거나 에러가 발생했습니다.")

✅ 삼성전자 DART 고유번호: 00126380
🚀 2025년도 삼성전자 재무제표 수집 중...

✅ 수집 완료! 주요 재무 계정 내역:


,fs_nm,account_nm,thstrm_amount,frmtrm_amount
0,연결재무제표,유동자산,"247,684,612,000,000","227,062,266,000,000"
1,연결재무제표,비유동자산,"319,257,498,000,000","287,469,682,000,000"
2,연결재무제표,자산총계,"566,942,110,000,000","514,531,948,000,000"
3,연결재무제표,유동부채,"106,411,348,000,000","93,326,299,000,000"
4,연결재무제표,비유동부채,"24,210,425,000,000","19,013,579,000,000"
5,연결재무제표,부채총계,"130,621,773,000,000","112,339,878,000,000"
6,연결재무제표,자본금,"897,514,000,000","897,514,000,000"
7,연결재무제표,이익잉여금,"402,135,600,000,000","370,513,188,000,000"
8,연결재무제표,자본총계,"436,320,337,000,000","402,192,070,000,000"
9,연결재무제표,매출액,"333,605,938,000,000","300,870,903,000,000"


In [2]:
# 쉼표(,) 제거 및 숫자(float) 타입으로 변환하는 전처리 함수
def clean_amount(value):
    if pd.isna(value):
        return 0
    return float(str(value).replace(',', ''))

# 당기(2025년) 부채총계와 자본총계 추출
# account_nm이 '부채총계'인 행을 찾아 thstrm_amount 값 가져오기
total_liability_raw = samsung_finstate.loc[samsung_finstate['account_nm'] == '부채총계', 'thstrm_amount'].values[0]
total_equity_raw = samsung_finstate.loc[samsung_finstate['account_nm'] == '자본총계', 'thstrm_amount'].values[0]

# 전처리 적용
total_liability = clean_amount(total_liability_raw)
total_equity = clean_amount(total_equity_raw)

# 부채비율 계산 (%)
debt_ratio = (total_liability / total_equity) * 100

print(f"✅ 삼성전자 당기 부채총계: {total_liability:,.0f} 원")
print(f"✅ 삼성전자 당기 자본총계: {total_equity:,.0f} 원")
print(f"🚨 삼성전자 부채비율: {debt_ratio:.2f}%")

✅ 삼성전자 당기 부채총계: 130,621,773,000,000 원
✅ 삼성전자 당기 자본총계: 436,320,337,000,000 원
🚨 삼성전자 부채비율: 29.94%


In [3]:
# 영업이익 당기(thstrm) 및 전기(frmtrm) 금액 추출
op_profit_row = samsung_finstate[samsung_finstate['account_nm'] == '영업이익']

current_op = clean_amount(op_profit_row['thstrm_amount'].values[0])
previous_op = clean_amount(op_profit_row['frmtrm_amount'].values[0])

# 영업이익 증감률(YoY) 계산 (%)
# 수식: (당기 - 전기) / abs(전기) * 100
if previous_op != 0:
    op_growth_rate = ((current_op - previous_op) / abs(previous_op)) * 100
else:
    op_growth_rate = 0

print(f"✅ 전기 영업이익: {previous_op:,.0f} 원")
print(f"✅ 당기 영업이익: {current_op:,.0f} 원")
print(f"📉 영업이익 증감률(YoY): {op_growth_rate:.2f}%\n")

# Mini 재무 Risk Scoring (Rule-based v1.0)
risk_score = 0
risk_reasons = []

# 조건 1: 부채비율 200% 초과 시 위험점수 30점 부여
if debt_ratio > 200:
    risk_score += 30
    risk_reasons.append("부채비율 200% 초과")

# 조건 2: 영업이익이 전년 대비 감소(-%) 시 위험점수 30점 부여
if op_growth_rate < 0:
    risk_score += 30
    risk_reasons.append("영업이익 전년 대비 감소")
    
# 조건 3: 영업이익이 '적자(0 미만)'인 경우 치명적이므로 추가점수 40점 부여
if current_op < 0:
    risk_score += 40
    risk_reasons.append("영업이익 적자 발생")

print("📊 [재무 리스크 평가 결과]")
print(f"총 Risk Score: {risk_score}점 / 100점")
if risk_reasons:
    print(f"위험 사유: {', '.join(risk_reasons)}")
else:
    print("위험 사유: 없음 (안전)")

✅ 전기 영업이익: 32,725,961,000,000 원
✅ 당기 영업이익: 43,601,051,000,000 원
📉 영업이익 증감률(YoY): 33.23%

📊 [재무 리스크 평가 결과]
총 Risk Score: 0점 / 100점
위험 사유: 없음 (안전)
